In [ ]:
# ============================================================================
# 1. INSTALLATION & SETUP
# ============================================================================
!pip cache purge # Clear pip cache to prevent corrupted metadata issues
!pip uninstall -y google-generativeai google-genai # Uninstall existing versions to prevent conflicts
!pip install -q -U --force-reinstall google-generativeai # Force reinstall google.generativeai to ensure it's installed correctly and is up-to-date

print("--- After google-generativeai reinstall ---")
try:
    import pkg_resources
    dist = pkg_resources.get_distribution('google-generativeai')
    print(f"Installed google-generativeai version: {dist.version}")
    print(f"Installed google-generativeai location: {dist.location}")
except Exception as e:
    print(f"Could not verify google-generativeai installation: {e}")
print("-------------------------------------")

!pip install -q -U pypdf streamlit gradio # Install other required libraries

import gradio as gr
import google.generativeai as genai # Reverted import to google.generativeai
from google.colab import userdata
import pypdf
import logging
import os
from logging.handlers import TimedRotatingFileHandler
import streamlit as st
import sys

# Configure basic logging to a file with rotation
log_dir = './logs'
os.makedirs(log_dir, exist_ok=True)

# Clear any existing handlers to ensure our new setup is effective
logging.getLogger().handlers = []

# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Create a rotating file handler
log_file_path = os.path.join(log_dir, 'note_genie.log')
handler = TimedRotatingFileHandler(log_file_path, when='midnight', interval=1, backupCount=7)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
logger.addHandler(handler)

logging.info("✅ NoteGenie: Logging configured with daily rotation.")

# Define the API secret name here
API_SECRET_NAME = 'GenieKey'

# Attempt to load your secret key and set it as an environment variable for google.generativeai
try:
    API_KEY = userdata.get(API_SECRET_NAME)
    os.environ['GENAI_API_KEY'] = API_KEY # Set API key as environment variable for google.generativeai
    logging.info("✅ NoteGenie: Gemini API Key successfully loaded and set as environment variable!")
except Exception as e:
    logging.error(f"❌ FATAL ERROR: '{API_SECRET_NAME}' not found in Colab Secrets or other API Key error: {e}. Exiting program.")
    sys.exit(1)

print(f"DEBUG: genai module path after import: {genai.__file__}")
print(f"DEBUG: 'GenerativeModel' in dir(genai) BEFORE exit check: {'GenerativeModel' in dir(genai)}")
print(f"DEBUG: dir(genai) BEFORE exit check: {dir(genai)}")

# Add a check for GenerativeModel after installation
if not hasattr(genai, 'GenerativeModel'):
    logging.error("❌ FATAL ERROR: 'google.generativeai' module does not have 'GenerativeModel' attribute after installation. Exiting program.")
    sys.exit(1)
else:
    logging.info("✅ NoteGenie: 'GenerativeModel' attribute found in 'google.generativeai'.")

logging.debug(f"Log file path for rotation: {os.path.abspath(log_file_path)}")

Files removed: 208
Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.3/173.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Test if the API key is correctly configured by checking the environment variable
import os
try:
    if 'GENAI_API_KEY' in os.environ and os.environ['GENAI_API_KEY']:
        print("✅ Gemini API key found in environment variable 'GENAI_API_KEY'.")
    else:
        print("❌ Gemini API key not found in environment variable 'GENAI_API_KEY'. Check setup.")
except Exception as e:
    print(f"❌ Error checking GENAI_API_KEY environment variable: {e}")

✅ Gemini API key found in environment variable 'GENAI_API_KEY'.


In [ ]:
import google.generativeai as genai
import logging

logging.info(f"Inspecting genai module after installation:")
logging.info(f"genai.__file__: {genai.__file__}")
logging.info(f"'configure' in dir(genai): {'configure' in dir(genai)}")
if not ('configure' in dir(genai)):
    logging.error("FATAL: 'configure' attribute is missing from google.genai. This indicates a severe installation or environment issue.")
else:
    logging.info("'configure' attribute found in google.genai.")

# Also check the version
try:
    from importlib.metadata import version
    genai_version = version('google-generativeai') # Check for the actual package name
    logging.info(f"Installed google-generativeai version: {genai_version}")
except Exception as e:
    logging.warning(f"Could not determine google-generativeai version: {e}")

In [ ]:

# ============================================================================
# 2. CONFIGURATION
# ============================================================================
class NoteGenieConfig:
    MODELS = {"fast": "gemini-flash-latest", "quality": "gemini-pro-latest"}
    STUDY_PROMPT = """You are NoteGenie. Generate a list of concise, relevant, and thoughtful questions from the following text that can be used for self-quizzing or study purposes. Focus on key concepts and details. Use **Bold** for important terms in the questions.

    TEXT: {text}"""

In [ ]:
# ============================================================================
# 3. CORE LOGIC
# ============================================================================
import os
import logging # Import logging if not already present from cell 1

class NoteGenie:
    def __init__(self):
        self.config = NoteGenieConfig()
        self.upload_dir = "./uploads"
        os.makedirs(self.upload_dir, exist_ok=True)
        logging.info(f"Upload directory created at: {self.upload_dir}")

    def list_uploaded_files(self):
        """Lists files in the uploads directory."""
        files = [f for f in os.listdir(self.upload_dir) if os.path.isfile(os.path.join(self.upload_dir, f))]
        return files

    def generate(self, text_input, selected_file_name, model_choice):
        text_to_process = ""

        # Prioritize selected file from uploads folder
        if selected_file_name and selected_file_name != "None (direct text input)": # Updated condition for Streamlit dropdown
            file_path = os.path.join(self.upload_dir, selected_file_name)
            if not os.path.exists(file_path):
                logging.error(f"Selected file not found: {file_path}")
                return f"❌ Error: Selected file '{selected_file_name}' not found in uploads folder."
            try:
                if selected_file_name.lower().endswith('.pdf'):
                    reader = pypdf.PdfReader(file_path)
                    text_to_process = "".join([page.extract_text() for page in reader.pages])
                    logging.info(f"Extracted text from PDF file: {selected_file_name}")
                else:
                    with open(file_path, "r", encoding="utf-8") as f:
                        text_to_process = f.read()
                    logging.info(f"Read text from file: {selected_file_name}")
            except Exception as e:
                logging.error(f"Error reading file {selected_file_name}: {str(e)}")
                return f"❌ Error reading file '{selected_file_name}': {str(e)}"
        elif text_input and text_input.strip(): # Use direct text input if no file selected or "None" selected
            text_to_process = text_input
            logging.info("Processing direct text input.")

        if not text_to_process or not text_to_process.strip():
            logging.warning("No text entered or file selected.")
            return "⚠️ Please enter some text or select a file from uploads folder!"

        try:
            # Removed gradio.Progress() dependency
            model = genai.GenerativeModel(model_name=self.config.MODELS[model_choice])
            logging.info(f"Generating content with model: {model_choice}")
            response = model.generate_content(self.config.STUDY_PROMPT.format(text=text_to_process))
            logging.info("Content generation successful.")
            return response.text.strip()
        except Exception as e:
            logging.error(f"Error during content generation: {str(e)}")
            return f"❌ Error: {str(e)}"

In [ ]:
# ============================================================================
# 4. STREAMLIT INTERFACE
# ============================================================================
def create_streamlit_ui():
    st.set_page_config(page_title="NoteGenie", layout="centered")

    app = NoteGenie()

    st.title("📚 NoteGenie")
    st.markdown("### Private AI Study Assistant")
    st.markdown("**Instructions:** Paste your study material in the text box below, or select a file from the 'uploads' folder. Select your preferred model, then click 'Generate' to get a hierarchical study guide.")

    # Input for text
    input_text = st.text_area("Paste Notes Here", height=150)

    # Dropdown for uploaded files
    uploaded_files = app.list_uploaded_files()
    uploaded_files_options = ["None (direct text input)"] + uploaded_files # Add option for direct text
    selected_file_name = st.selectbox("Or Select File from Uploads Folder", uploaded_files_options)

    # Model selection
    model_choice = st.selectbox("Select Model", list(app.config.MODELS.keys()), index=0)

    # Generate button
    if st.button("✨ Generate"): # Assign a key to the button if there are multiple buttons on the page.
        if not input_text.strip() and selected_file_name == "None (direct text input)":
            st.warning("⚠️ Please enter some text or select a file from the uploads folder!")
        else:
            with st.spinner("Consulting Gemini..."):
                output = app.generate(input_text, selected_file_name, model_choice)
                st.text_area("Study Guide", value=output, height=400)

# This will be called when the Streamlit app is run
if __name__ == "__main__":
    create_streamlit_ui()

# Save this cell's content to a file to be run by Streamlit
with open('streamlit_app.py', 'w') as f:
    f.write('''
import streamlit as st
import google.generativeai as genai # Reverted import to google.generativeai
# Removed from google.colab import userdata, as it's not available in standalone Streamlit
import pypdf
import logging
import os
from logging.handlers import TimedRotatingFileHandler

# Re-import necessary components from other cells if they are not in `streamlit_app.py`
# Assuming NoteGenieConfig and NoteGenie class definitions are available (e.g., copied or imported from another file)
# For simplicity, copy the relevant class definitions here for standalone execution

# Configuration Class (from uwZhSTnAcnSB)
class NoteGenieConfig:
    MODELS = {"fast": "gemini-flash-latest", "quality": "gemini-pro-latest"}
    STUDY_PROMPT = """You are NoteGenie. Generate a list of concise, relevant, and thoughtful questions from the following text that can be used for self-quizzing or study purposes. Focus on key concepts and details. Use **Bold** for important terms in the questions.\n\n    TEXT: {text}"""

# NoteGenie Class (from rMDDQl9ycupH)
class NoteGenie:
    def __init__(self):
        self.config = NoteGenieConfig()
        self.upload_dir = ".//uploads"
        os.makedirs(self.upload_dir, exist_ok=True)
        # Logging setup is typically done once globally, so we won't duplicate it here
        # logging.info(f"Upload directory created at: {self.upload_dir}") # Avoid re-logging during app execution

    def list_uploaded_files(self):
        """Lists files in the uploads directory."""
        files = [f for f in os.listdir(self.upload_dir) if os.path.isfile(os.path.join(self.upload_dir, f))]
        return files

    def generate(self, text_input, selected_file_name, model_choice):
        text_to_process = ""

        if selected_file_name and selected_file_name != "None (direct text input)":
            file_path = os.path.join(self.upload_dir, selected_file_name)
            if not os.path.exists(file_path):
                # Use st.error for Streamlit UI feedback
                st.error(f"❌ Error: Selected file '{selected_file_name}' not found in uploads folder.")
                return f"❌ Error: Selected file '{selected_file_name}' not found in uploads folder."
            try:
                if selected_file_name.lower().endswith('.pdf'):
                    reader = pypdf.PdfReader(file_path)
                    text_to_process = "".join([page.extract_text() for page in reader.pages])
                else:
                    with open(file_path, "r", encoding="utf-8") as f:
                        text_to_process = f.read()
            except Exception as e:
                st.error(f"❌ Error reading file '{selected_file_name}': {str(e)}")
                return f"❌ Error reading file '{selected_file_name}': {str(e)}"
        elif text_input and text_input.strip():
            text_to_process = text_input

        if not text_to_process or not text_to_process.strip():
            st.warning("⚠️ Please enter some text or select a file from uploads folder!")
            return "⚠️ Please enter some text or select a file from uploads folder!"

        try:
            model = genai.GenerativeModel(model_name=self.config.MODELS[model_choice])
            response = model.generate_content(self.config.STUDY_PROMPT.format(text=text_to_process))
            return response.text.strip()
        except Exception as e:
            st.error(f"❌ Error during content generation: {str(e)}")
            return f"❌ Error: {str(e)}"

# Main Streamlit UI function
def create_streamlit_ui():
    st.set_page_config(page_title="NoteGenie", layout="centered")

    app = NoteGenie()

    st.title("📚 NoteGenie")
    st.markdown("### Private AI Study Assistant")
    st.markdown("**Instructions:** Paste your study material in the text box below, or select a file from the 'uploads' folder. Select your preferred model, then click 'Generate' to get a hierarchical study guide.")

    input_text = st.text_area("Paste Notes Here", height=150)

    uploaded_files = app.list_uploaded_files()
    uploaded_files_options = ["None (direct text input)"] + uploaded_files
    selected_file_name = st.selectbox("Or Select File from Uploads Folder", uploaded_files_options)

    model_choice = st.selectbox("Select Model", list(app.config.MODELS.keys()), index=0)

    if st.button("✨ Generate"):
        if not input_text.strip() and selected_file_name == "None (direct text input)":
            st.warning("⚠️ Please enter some text or select a file from the uploads folder!")
        else:
            with st.spinner("Consulting Gemini..."):
                output = app.generate(input_text, selected_file_name, model_choice)
                st.text_area("Study Guide", value=output, height=400)

if __name__ == "__main__":
    # Try to get API_KEY from environment variable (set by Colab host)
    api_key_from_env = os.environ.get('GENAI_API_KEY')
    if api_key_from_env:
        try:
            genai.configure(api_key=api_key_from_env) # Explicitly configure the API key
            # DEBUG statements
            st.info(f"DEBUG: genai module path: {genai.__file__}")
            st.info(f"DEBUG: 'configure' in dir(genai): {'configure' in dir(genai)}")
            # No genai.configure() needed here, as google.generativeai relies on environment variables
        except Exception as e:
            st.error(f"❌ ERROR handling API Key from environment variable in Streamlit: {e}")
            st.stop()
    else:
        # If GENAI_API_KEY is not set, it's an error in this context, as userdata.get() won't work in subprocess.
        st.error("❌ ERROR: 'GenieKey' not found in environment variables. Streamlit app cannot configure Gemini API.")
        st.stop()

    create_streamlit_ui()
'''
)


2026-04-22 17:21:03.907 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 17:21:03.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 17:21:03.910 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 17:21:03.911 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 17:21:03.912 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 17:21:03.912 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 17:21:03.913 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 17:21:03.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
import os

# Path to the log file
log_file_path = 'output.log'

# Check if the log file exists and print its content
if os.path.exists(log_file_path):
    with open(log_file_path, 'r') as f:
        log_content = f.read()
    print(log_content)
else:
    print(f"Log file not found at: {log_file_path}")





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://130.211.227.98:8501

/content/streamlit_app.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai # Reverted import to google.generativeai
  Stopping...



In [ ]:

# ============================================================================
# 2. CONFIGURATION
# ============================================================================
class NoteGenieConfig:
    MODELS = {"fast": "gemini-1.5-flash", "quality": "gemini-1.5-pro"}
    STUDY_PROMPT = """You are NoteGenie. Convert the following text into clear,
    hierarchical bullet points. Use **Bold** for key terms.

    TEXT: {text}"""

In [ ]:
import os

log_dir = './logs'
log_file_path = os.path.join(log_dir, 'note_genie.log')

if os.path.exists(log_file_path):
    with open(log_file_path, 'r') as f:
        log_content = f.read()
    print(log_content)
else:
    print(f"Log file not found at: {log_file_path}")

2026-04-22 15:44:05,286 - INFO - ✅ NoteGenie: Logging configured with daily rotation.
2026-04-22 15:44:08,225 - INFO - ✅ NoteGenie: Gemini API Key successfully loaded!
2026-04-22 15:44:08,226 - DEBUG - Log file path for rotation: /content/logs/note_genie.log
2026-04-22 15:44:08,265 - INFO - Upload directory created at: ./uploads
2026-04-22 15:46:03,952 - INFO - ✅ NoteGenie: Logging configured with daily rotation.
2026-04-22 15:46:04,370 - ERROR - ❌ ERROR: 'GenieKey' not found in Colab Secrets or other API Key error: module 'google.genai' has no attribute 'configure'
2026-04-22 15:46:04,370 - DEBUG - Log file path for rotation: /content/logs/note_genie.log
2026-04-22 15:46:20,714 - INFO - ✅ NoteGenie: Logging configured with daily rotation.
2026-04-22 15:46:21,208 - ERROR - ❌ ERROR: 'GenieKey' not found in Colab Secrets or other API Key error: module 'google.genai' has no attribute 'configure'
2026-04-22 15:46:21,209 - DEBUG - Log file path for rotation: /content/logs/note_genie.log
202

In [ ]:
print('Listing available Gemini models:')
for model in genai.list_models():
    print(f"  Name: {model.name}, Supported operations: {model.supported_generation_methods}")

Listing available Gemini models:


2026-04-22 17:21:05.109 200 GET /v1beta/models?pageSize=50&%24alt=json%3Benum-encoding%3Dint (::1) 1140.62ms


  Name: models/gemini-2.5-flash, Supported operations: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Name: models/gemini-2.5-pro, Supported operations: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Name: models/gemini-2.0-flash, Supported operations: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Name: models/gemini-2.0-flash-001, Supported operations: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Name: models/gemini-2.0-flash-lite-001, Supported operations: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Name: models/gemini-2.0-flash-lite, Supported operations: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Name: models/gemini-2.5-flash-preview-tts, Supported operations: ['countTokens', 'generateContent']
  Name: models/gemini-2.5-pro-preview-tts, Supported operati

2026-04-22 17:21:05.557 200 GET /v1beta/models?pageSize=50&pageToken=CiRtb2RlbHMvdmVvLTMuMS1mYXN0LWdlbmVyYXRlLXByZXZpZXc%3D&%24alt=json%3Benum-encoding%3Dint (::1) 431.93ms


  Name: models/veo-3.1-lite-generate-preview, Supported operations: ['predictLongRunning']
  Name: models/gemini-2.5-flash-native-audio-latest, Supported operations: ['countTokens', 'bidiGenerateContent']
  Name: models/gemini-2.5-flash-native-audio-preview-09-2025, Supported operations: ['countTokens', 'bidiGenerateContent']
  Name: models/gemini-2.5-flash-native-audio-preview-12-2025, Supported operations: ['countTokens', 'bidiGenerateContent']
  Name: models/gemini-3.1-flash-live-preview, Supported operations: ['bidiGenerateContent']


In [ ]:
# ============================================================================
# 5. RUN STREAMLIT APP
# ============================================================================
# Install localtunnel
!npm install -g localtunnel

# Terminate any existing processes on port 8501
!kill -9 $(lsof -t -i:8501) >/dev/null 2>&1 || true

# Get API key from Colab secrets in the main Colab process
COLAB_API_KEY = None # Initialize COLAB_API_KEY
try:
    # Assuming API_SECRET_NAME is defined in an earlier cell (e.g., WGL1aeFiYFCH)
    # If not, it would need to be defined or re-imported here.
    # For robustness, we'll try to use it if it exists, otherwise default to 'GenieKey'
    if 'API_SECRET_NAME' in globals():
        COLAB_API_KEY = userdata.get(globals()['API_SECRET_NAME'])
    else:
        COLAB_API_KEY = userdata.get('GenieKey') # Fallback if API_SECRET_NAME isn't available

    if COLAB_API_KEY:
        print("✅ API Key loaded from Colab secrets.")
    else:
        print("⚠️ Warning: 'GenieKey' not found in Colab Secrets. Streamlit app might fail without an API key.")
except Exception as e:
    print(f"❌ Error loading 'GenieKey' from Colab Secrets: {e}")

# Run Streamlit in the background, explicitly passing the API key as an environment variable
if COLAB_API_KEY:
    # Use bash -c to properly parse environment variable assignment with nohup
    !nohup bash -c 'GENAI_API_KEY="{COLAB_API_KEY}" python -m streamlit run streamlit_app.py --server.port 8501 --server.enableCORS True --server.enableXsrfProtection False > output.log 2>&1' &
    print("✅ Streamlit app launched with API key passed as environment variable.")
else:
    print("❌ Streamlit app not launched because API key could not be loaded.")

# Wait a moment for Streamlit to start
import time
time.sleep(5)

# Expose port 8501 with localtunnel and print the public URL
print('Streamlit App is live at:')
!lt --port 8501


In [ ]:
# ============================================================================
# 5. RUN STREAMLIT APP
# ============================================================================
# Install localtunnel
!npm install -g localtunnel

# Terminate any existing processes on port 8501
!kill -9 $(lsof -t -i:8501) >/dev/null 2>&1 || true

# Get API key from Colab secrets in the main Colab process
COLAB_API_KEY = None # Initialize COLAB_API_KEY
try:
    # Assuming API_SECRET_NAME is defined in an earlier cell (e.g., WGL1aeFiYFCH)
    # If not, it would need to be defined or re-imported here.
    # For robustness, we'll try to use it if it exists, otherwise default to 'GenieKey'
    if 'API_SECRET_NAME' in globals():
        COLAB_API_KEY = userdata.get(globals()['API_SECRET_NAME'])
    else:
        COLAB_API_KEY = userdata.get('GenieKey') # Fallback if API_SECRET_NAME isn't available

    if COLAB_API_KEY:
        print("✅ API Key loaded from Colab secrets.")
    else:
        print("⚠️ Warning: 'GenieKey' not found in Colab Secrets. Streamlit app might fail without an API key.")
except Exception as e:
    print(f"❌ Error loading 'GenieKey' from Colab Secrets: {e}")

# Run Streamlit in the background, explicitly passing the API key as an environment variable
if COLAB_API_KEY:
    # Use bash -c to properly parse environment variable assignment with nohup
    !nohup bash -c 'GENAI_API_KEY="{COLAB_API_KEY}" python -m streamlit run streamlit_app.py --server.port 8501 --server.enableCORS True --server.enableXsrfProtection False > output.log 2>&1' &
    print("✅ Streamlit app launched with API key passed as environment variable.")
else:
    print("❌ Streamlit app not launched because API key could not be loaded.")

# Wait a moment for Streamlit to start
import time
time.sleep(5)

# Expose port 8501 with localtunnel and print the public URL
print('Streamlit App is live at:')
!lt --port 8501


⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 1s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇✅ API Key loaded from Colab secrets.
nohup: appending output to 'nohup.out'
✅ Streamlit app launched with API key passed as environment variable.
Streamlit App is live at:
your url is: https://plenty-teams-trade.loca.lt


In [ ]:
import os

# Path to the Streamlit app's output log file
streamlit_log_file_path = 'output.log'

# Check if the log file exists and print its content
if os.path.exists(streamlit_log_file_path):
    with open(streamlit_log_file_path, 'r') as f:
        log_content = f.read()
    print(log_content)
else:
    print(f"Streamlit log file not found at: {streamlit_log_file_path}")

In [ ]:
import os

# Path to the Streamlit app file
streamlit_app_file_path = 'streamlit_app.py'

# Check if the file exists and print its content
if os.path.exists(streamlit_app_file_path):
    with open(streamlit_app_file_path, 'r') as f:
        app_content = f.read()
    print("--- Content of streamlit_app.py ---")
    print(app_content)
    print("-----------------------------------")
else:
    print(f"Streamlit app file not found at: {streamlit_app_file_path}")